# Lab 4: Live Deep Agent Professor Workspace

This lab teaches **Deep Agents** as the next abstraction after Lab 3:

- Lab 3 used a **swarm** to hand control between specialists.
- Lab 4 uses **one deep agent** to manage an open-ended, file-based task.
- The deep agent answers broad questions from a compact index, opens full dossiers only when needed, and delegates one live add workflow to a subagent.

We keep the core lesson focused:

- no Neo4j
- no custom `StateGraph`
- no swarm router
- normal question-answering stays grounded in local files
- live crawl + OCR only appears in the add workflow

The sandbox knowledge base lives in files:

- `professors.md` is the compact index
- `professors/<slug>.md` stores the sandbox dossier
- `skills/add-professor-from-url/SKILL.md` stores the live add workflow

The committed source corpus still lives under `lab_4_deep_agents/professors/`, but this notebook now treats the current `/sandbox` workspace as the active Lab 4 dataset. On reruns it refreshes structural files like `professors.md` and the live URL skill in place without restoring deleted dossier files. Broad QA stays local and deterministic, while the add flow shows how a deep agent can use one tightly constrained `execute` path for the sanctioned add script.

**Learn more:** [Deep Agents overview](https://docs.langchain.com/oss/python/deepagents/overview), [customization](https://docs.langchain.com/oss/python/deepagents/customization), [skills](https://docs.langchain.com/oss/python/deepagents/skills), [subagents](https://docs.langchain.com/oss/python/deepagents/subagents)


## 0. Big Picture

![Lab 4 architecture overview](teaching_diagrams/05_lab4_live_deep_agent_workspace.png)

Use this diagram as the mental model for the rest of the lab: one main deep agent, one delegated add subagent, one sandboxed backend, and one shared chat surface.


## 1. Environment Check

Learning goal: verify that this notebook is running inside the project environment and that `deepagents` is installed.

This lab now uses a notebook-local sandboxed Deep Agents backend, so the live add flow requires two things in addition to the normal chat model:

- working OCR settings in `.env`
- live network access to the official BIT detail page during the add workflow


In [1]:
from __future__ import annotations

import json
import os
import re
import shutil
import sys
import warnings
from importlib.metadata import version
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
        if (candidate / "pyproject.toml").exists():
            PROJECT_ROOT = candidate
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore", message="IProgress not found.*")

from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend
from deepagents.backends.protocol import ExecuteResponse
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

from bit_professor_chat.config import TutorSettings
from bit_professor_chat.legacy_cache import build_cached_summary_line, extract_markdown_section_lines
from bit_professor_chat.model_factory import build_model


def pretty(data: Any) -> None:
    print(json.dumps(data, ensure_ascii=False, indent=2))


def slug_to_display_name(slug: str) -> str:
    return " ".join(part.capitalize() for part in slug.split("-") if part)


settings = TutorSettings.from_env(PROJECT_ROOT / ".env")
ocr_config = settings.require_ocr()
model = build_model(settings)

LAB_ROOT = PROJECT_ROOT / "lab_4_deep_agents"
SOURCE_DOSSIER_DIR = LAB_ROOT / "professors"
SOURCE_CORE_SKILL_DIR = LAB_ROOT / "skills" / "add-professor-from-url"
SANDBOX_ROOT = LAB_ROOT / "sandbox"
EXAMPLE_ADD_NAME = "Tang Haijing"
EXAMPLE_ADD_DETAIL_URL = "https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b121631.htm"


def build_shell_env() -> dict[str, str]:
    path_entries = [str(PROJECT_ROOT / ".venv" / "bin")]
    existing_path = os.getenv("PATH")
    if existing_path:
        path_entries.append(existing_path)

    shell_env = {
        "PATH": ":".join(dict.fromkeys(entry for entry in path_entries if entry)),
        "PYTHONPATH": str(PROJECT_ROOT),
        "BIT_PROF_LLM_API_KEY": settings.llm_api_key,
        "BIT_PROF_LLM_BASE_URL": settings.llm_base_url,
        "BIT_PROF_LLM_MODEL": settings.llm_model,
        "BIT_PROF_OCR_API_KEY": ocr_config["api_key"],
        "BIT_PROF_OCR_BASE_URL": ocr_config["base_url"],
        "BIT_PROF_OCR_MODEL": ocr_config["model"],
    }
    home = os.getenv("HOME")
    if home:
        shell_env["HOME"] = home
    return shell_env


SHELL_ENV = build_shell_env()
source_dossier_preview = [path.stem for path in sorted(SOURCE_DOSSIER_DIR.glob("*.md"))[:10]]

pretty(
    {
        "project_root": str(PROJECT_ROOT),
        "source_dossier_dir": str(SOURCE_DOSSIER_DIR),
        "base_url": settings.lab_tutor_llm_base_url,
        "model": settings.lab_tutor_llm_model,
        "ocr_model": ocr_config["model"],
        "recommended_base_url": "https://api.silra.cn/v1/",
        "recommended_model": "qwen3.5-plus",
        "deepagents_version": version("deepagents"),
        "lab_root": str(LAB_ROOT),
        "sandbox_root": str(SANDBOX_ROOT),
        "sandbox_virtual_root": "/",
        "source_dossier_count": len(list(SOURCE_DOSSIER_DIR.glob("*.md"))),
        "source_dossier_preview": source_dossier_preview,
        "example_add_name": EXAMPLE_ADD_NAME,
        "example_add_detail_url": EXAMPLE_ADD_DETAIL_URL,
        "shell_backend": "Lab4SandboxBackend(LocalShellBackend)",
    }
)


{
  "project_root": "/Users/khajievroma/Projects/agents_tutorial",
  "source_dossier_dir": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents/professors",
  "base_url": "https://api.silra.cn/v1/",
  "model": "qwen3.5-plus",
  "ocr_model": "qwen-vl-ocr-latest",
  "recommended_base_url": "https://api.silra.cn/v1/",
  "recommended_model": "qwen3.5-plus",
  "deepagents_version": "0.5.2",
  "lab_root": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents",
  "sandbox_root": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents/sandbox",
  "sandbox_virtual_root": "/",
  "source_dossier_count": 44,
  "source_dossier_preview": [
    "che-haiying",
    "cheng-cheng",
    "filippo-fabrocini",
    "gao-guangyu",
    "gao-yang",
    "huang-heyan",
    "huang-yonggang",
    "jin-fusheng",
    "li-dongni",
    "li-fan"
  ],
  "example_add_name": "Tang Haijing",
  "example_add_detail_url": "https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b121631.htm",
  "shell_ba

## 2. Why Deep Agents?

Learning goal: make the abstraction jump explicit before we write any code.

Theory: a basic agent is good for a short tool loop. A custom LangGraph is good when you already know the control flow. A swarm is good when conversational ownership should move between specialists. A deep agent is better when the task is open-ended, file-centric, and easier to express as policy plus tools instead of a hand-built graph.


In [2]:
pretty(
    {
        "basic_create_agent": "short bounded tool loop",
        "custom_langgraph": "explicit graph when you know the workflow shape",
        "swarm": "specialists hand off conversational ownership",
        "deep_agent": "one harness with files, skills, delegated subagents, and execute",
        "lab_4_focus": "prompt rules + file tools + one skill-owned script",
    }
)


{
  "basic_create_agent": "short bounded tool loop",
  "custom_langgraph": "explicit graph when you know the workflow shape",
  "swarm": "specialists hand off conversational ownership",
  "deep_agent": "one harness with files, skills, delegated subagents, and execute",
  "lab_4_focus": "prompt rules + file tools + one skill-owned script"
}


## 3. Prepare the Sandbox Workspace

Learning goal: prepare the current sandbox workspace in place so the deep agent uses the files that are already there.

Theory: the sandbox root is the deep agent's file world. On reruns we keep the current sandbox dossier folder, recreate any missing structural directories, refresh the live URL skill, and rebuild a fresh compact index. Inside the agent, `/` now means the sandbox root, not your laptop root. That keeps ordinary QA deterministic while letting manual sandbox edits, including dossier deletions, persist across reruns.


In [3]:
def publication_title_from_line(line: str) -> str:
    return line.split("; ", 1)[0].strip().rstrip(".")


def summarize_professor_markdown(markdown_text: str, display_name: str) -> str:
    research_lines = extract_markdown_section_lines(markdown_text, ("Research Interests",))
    if research_lines:
        return f"{display_name}: {', '.join(research_lines[:3])}"

    publication_titles = [
        publication_title_from_line(line)
        for line in extract_markdown_section_lines(markdown_text, ("Publications",))
    ]
    publication_titles = [title for title in publication_titles if title]
    if publication_titles:
        return f"{display_name}: {', '.join(publication_titles[:3])}"

    return build_cached_summary_line(markdown_text, display_name)


def rebuild_runtime_index(workspace_root: Path) -> list[str]:
    # Rebuild the sandbox index with a compact, readable summary per professor.
    dossier_dir = workspace_root / "professors"
    indexed_professors: list[str] = []
    lines = ["# BIT CSAT Professors", ""]

    for markdown_path in sorted(dossier_dir.glob("*.md")):
        markdown_text = markdown_path.read_text(encoding="utf-8")
        display_name = slug_to_display_name(markdown_path.stem)
        lines.append(f"- {summarize_professor_markdown(markdown_text, display_name)}")
        indexed_professors.append(display_name)

    lines.append("")
    index_path = workspace_root / "professors.md"
    index_path.write_text("\n".join(lines), encoding="utf-8")
    return indexed_professors


def prepare_runtime_workspace() -> dict[str, Any]:
    # Keep the current sandbox dossier set, but restore the structural paths Lab 4 needs.
    professors_dir = SANDBOX_ROOT / "professors"
    incoming_dir = SANDBOX_ROOT / "incoming"
    skills_root = SANDBOX_ROOT / "skills"
    skill_dir = skills_root / "add-professor-from-url"

    professors_dir.mkdir(parents=True, exist_ok=True)
    incoming_dir.mkdir(parents=True, exist_ok=True)
    skills_root.mkdir(parents=True, exist_ok=True)

    shutil.copytree(
        SOURCE_CORE_SKILL_DIR,
        skill_dir,
        dirs_exist_ok=True,
    )

    indexed_professors = rebuild_runtime_index(SANDBOX_ROOT)
    index_preview = (SANDBOX_ROOT / "professors.md").read_text(encoding="utf-8").splitlines()[:10]
    dossier_paths = sorted(professors_dir.glob("*.md"))

    return {
        "lab_root": str(LAB_ROOT),
        "sandbox_root": str(SANDBOX_ROOT),
        "sandbox_virtual_root": "/",
        "dossier_count": len(dossier_paths),
        "professor_count": len(indexed_professors),
        "sandbox_dossier_preview": [
            str(path.relative_to(PROJECT_ROOT))
            for path in dossier_paths[:10]
        ],
        "core_skill_path": "skills/add-professor-from-url/SKILL.md",
        "example_add_detail_url": EXAMPLE_ADD_DETAIL_URL,
        "index_preview": index_preview,
    }


runtime_snapshot = prepare_runtime_workspace()
pretty(runtime_snapshot)


{
  "lab_root": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents",
  "sandbox_root": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents/sandbox",
  "sandbox_virtual_root": "/",
  "dossier_count": 43,
  "index_line_count": 43,
  "sandbox_dossier_preview": [
    "lab_4_deep_agents/sandbox/professors/cheng-cheng.md",
    "lab_4_deep_agents/sandbox/professors/filippo-fabrocini.md",
    "lab_4_deep_agents/sandbox/professors/gao-guangyu.md",
    "lab_4_deep_agents/sandbox/professors/gao-yang.md",
    "lab_4_deep_agents/sandbox/professors/huang-heyan.md",
    "lab_4_deep_agents/sandbox/professors/huang-yonggang.md",
    "lab_4_deep_agents/sandbox/professors/jin-fusheng.md",
    "lab_4_deep_agents/sandbox/professors/li-dongni.md",
    "lab_4_deep_agents/sandbox/professors/li-fan.md",
    "lab_4_deep_agents/sandbox/professors/li-jianwu.md"
  ],
  "core_skill_path": "skills/add-professor-from-url/SKILL.md",
  "example_add_detail_url": "https://isc.bit.edu.cn/schools/csa

In [4]:
core_skill_preview = (
    SANDBOX_ROOT / "skills" / "add-professor-from-url" / "SKILL.md"
).read_text(encoding="utf-8")

print("\n".join(core_skill_preview.splitlines()[:32]))


---
name: add-professor-from-url
description: Use this skill when the task is to add one BIT professor from an official detail-page URL into the sandbox professor workspace.
---

# Add Professor From URL

## When To Use

Use this skill when:

- the user provides a BIT professor detail URL
- the user asks to add or refresh one professor from the official source page

Do not use this skill for normal question-answering over existing sandbox files.

## Goal

Create exactly one canonical dossier file under `professors/<slug>.md` from one official detail URL inside the sandbox workspace.

The parent agent is responsible for rebuilding `professors.md`. This skill only covers the dossier-creation workflow.

## Required Workflow

1. Treat `/` as the sandbox root and inspect `professors/*.md` first so you understand the current workspace.
2. Accept only an official BIT CSAT detail URL shaped like `https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b123456.htm`.
3. Run exactly one command wi

### Files vs Backend vs Sandboxing

These three ideas are related, but they are not the same:

- **Files** are the content the agent works with.
  In this lab that means `professors.md`, `professors/<slug>.md`, and `skills/add-professor-from-url/SKILL.md`.
- **Backend** is the runtime bridge that lets the agent read, write, search, and execute.
  In this lab that bridge is `Lab4SandboxBackend(LocalShellBackend)`.
- **Sandboxing** is the safety boundary around that backend.
  In this lab it means the sandbox root folder, virtual file paths, and a strict shell allowlist.

A good way to say it out loud in class:

- files are the agent's knowledge surface
- the backend is the agent's hands
- sandboxing is the fence around those hands


In [5]:
pretty(
    {
        "files_layer": {
            "SOURCE_DOSSIER_DIR": {
                "meaning": "committed professor corpus in the repo",
                "lab4_value": str(SOURCE_DOSSIER_DIR),
            },
            "SANDBOX_ROOT": {
                "meaning": "runtime workspace the agent is allowed to use",
                "lab4_value": str(SANDBOX_ROOT),
            },
            "professors.md": {
                "meaning": "compact index for broad lookup",
                "lab4_value": str(SANDBOX_ROOT / "professors.md"),
            },
            "professors/<slug>.md": {
                "meaning": "detailed dossier for deep lookup",
                "lab4_example": str(next(iter(sorted((SANDBOX_ROOT / "professors").glob("*.md"))), "")),
            },
            "skills/add-professor-from-url/SKILL.md": {
                "meaning": "workflow instructions for the add subagent",
                "lab4_value": str(SANDBOX_ROOT / "skills" / "add-professor-from-url" / "SKILL.md"),
            },
        }
    }
)


{
  "files_layer": {
    "SOURCE_DOSSIER_DIR": {
      "meaning": "committed professor corpus in the repo",
      "lab4_value": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents/professors"
    },
    "SANDBOX_ROOT": {
      "meaning": "runtime workspace the agent is allowed to use",
      "lab4_value": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents/sandbox"
    },
    "professors.md": {
      "meaning": "compact index for broad lookup",
      "lab4_value": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents/sandbox/professors.md"
    },
    "professors/<slug>.md": {
      "meaning": "detailed dossier for deep lookup",
      "lab4_example": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents/sandbox/professors/cheng-cheng.md"
    },
    "skills/add-professor-from-url/SKILL.md": {
      "meaning": "workflow instructions for the add subagent",
      "lab4_value": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents/sandbox/skil

## 4. Define the Deterministic Helper

Learning goal: keep the custom tool layer as small as possible.

Theory: the deep agent already knows how to read, write, list, and search files. In the core lesson we add exactly one deterministic helper: rebuild the compact index after a successful dossier add.


In [6]:
@tool
def rebuild_professors_index_tool() -> dict[str, Any]:
    """Rebuild the enriched professors.md from the current sandbox dossier folder."""
    indexed_professors = rebuild_runtime_index(SANDBOX_ROOT)
    index_path = SANDBOX_ROOT / "professors.md"
    return {
        "index_path": "professors.md",
        "professor_count": len(indexed_professors),
        "preview": index_path.read_text(encoding="utf-8").splitlines()[:12],
    }


pretty({"main_agent_custom_tools": [rebuild_professors_index_tool.name]})


{
  "main_agent_custom_tools": [
    "rebuild_professors_index_tool"
  ]
}


## 5. Build the Subagent and the Main Deep Agent

Learning goal: keep the orchestration pure Deep Agents.

Theory: the main agent uses built-in file tools plus one deterministic helper. The add workflow lives in one subagent, one skill, and one execute-backed script. That is the whole design: no custom graph, no router nodes, and no separate supervisor.


In [7]:
# Sandboxing layer: allow only the one documented add command to use shell execution.
ALLOWED_ADD_COMMAND = re.compile(
    r"^python skills/add-professor-from-url/scripts/add_professor_from_url\.py --detail-url (?:\"https://isc\.bit\.edu\.cn/schools/csat/knowingprofessors5/b\d+\.htm\"|'https://isc\.bit\.edu\.cn/schools/csat/knowingprofessors5/b\d+\.htm'|https://isc\.bit\.edu\.cn/schools/csat/knowingprofessors5/b\d+\.htm) --professors-dir professors$"
)


class Lab4SandboxBackend(LocalShellBackend):
    """Teach Deep Agents with a clear virtual file root and one sanctioned execute path."""

    def execute(self, command: str, *, timeout: int | None = None) -> ExecuteResponse:
        normalized = command.strip()
        if normalized == "python --version":
            return super().execute(normalized, timeout=timeout)
        if not ALLOWED_ADD_COMMAND.fullmatch(normalized):
            return ExecuteResponse(
                output=(
                    "Error: Lab 4 sandbox blocked this shell command. "
                    "Only the documented add-professor script is allowed."
                ),
                exit_code=1,
                truncated=False,
            )
        return super().execute(normalized, timeout=timeout)


# Backend layer: this object powers the built-in file tools and the execute tool.
# - root_dir: host folder that backs the sandbox
# - virtual_mode: file tools treat root_dir as `/`
# - timeout: default shell timeout in seconds
# - max_output_bytes: truncate oversized shell output
# - env: explicit environment visible to shell commands
# - inherit_env: when False, do not leak the whole host environment
BACKEND = Lab4SandboxBackend(
    root_dir=str(SANDBOX_ROOT),
    virtual_mode=True,
    timeout=300,
    max_output_bytes=100_000,
    env=SHELL_ENV,
    inherit_env=False,
)
CHECKPOINTER = InMemorySaver()

shell_probe = BACKEND.execute("python --version")

MAIN_SYSTEM_PROMPT = """
You are ProfessorWorkspaceAgent for a local professor knowledge base.

Rules:
- Work only inside the sandbox workspace.
- Inside this backend, `/` is the sandbox root, not the host filesystem root.
- Use sandbox paths like `/professors.md`, `/professors/`, `/skills/`, and `/incoming/`.
- Never request or invent absolute host filesystem paths.
- Keep answers grounded in the local files.
- For broad questions, start with /professors.md.
- For named-professor questions, start with /professors.md and open one dossier only if you need more detail.
- If the user asks to add a professor from an official BIT CSAT detail URL, delegate to AddProfessorSubagent.
- If AddProfessorSubagent returns status=added, call rebuild_professors_index_tool before replying.
- After a successful add and index rebuild, reply with a short confirmation that names the professor and the new dossier path.
- Do not expose internal file-tool errors, retries, or dead ends to the user. Recover silently and only give the user the final useful answer.
""".strip()

ADD_SUBAGENT_PROMPT = """
You are AddProfessorSubagent.

Use the add-professor-from-url skill for the exact workflow.

Rules:
- Treat `/` as the sandbox root.
- Inspect professors/ first.
- Accept only official BIT CSAT detail URLs.
- Use execute exactly once to run the skill-owned script.
- Do not rebuild professors.md.
- Return exactly one compact final line with status, professor_name, slug, markdown_path, and page_count.
""".strip()

add_professor_subagent = {
    "name": "AddProfessorSubagent",
    "description": "Use this agent when the user wants to add one professor from an official BIT CSAT detail URL into the sandbox workspace.",
    "system_prompt": ADD_SUBAGENT_PROMPT,
    "model": model,
    "tools": [],
    "skills": ["/skills/"],
}

professor_workspace_agent = create_deep_agent(
    model=model,
    tools=[rebuild_professors_index_tool],
    system_prompt=MAIN_SYSTEM_PROMPT,
    backend=BACKEND,
    subagents=[add_professor_subagent],
    checkpointer=CHECKPOINTER,
    name="ProfessorWorkspaceAgent",
)


pretty(
    {
        "lab_root": str(LAB_ROOT),
        "sandbox_root": str(SANDBOX_ROOT),
        "sandbox_virtual_root": "/",
        "virtual_mode": True,
        "core_skill_path": "skills/add-professor-from-url/SKILL.md",
        "example_add_detail_url": EXAMPLE_ADD_DETAIL_URL,
        "main_agent_name": "ProfessorWorkspaceAgent",
        "subagent_name": add_professor_subagent["name"],
        "shell_probe_output": shell_probe.output.strip(),
        "shell_probe_exit_code": shell_probe.exit_code,
    }
)


{
  "lab_root": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents",
  "sandbox_root": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents/sandbox",
  "sandbox_virtual_root": "/",
  "virtual_mode": true,
  "core_skill_path": "skills/add-professor-from-url/SKILL.md",
  "example_add_detail_url": "https://isc.bit.edu.cn/schools/csat/knowingprofessors5/b121631.htm",
  "main_agent_name": "ProfessorWorkspaceAgent",
  "subagent_name": "AddProfessorSubagent",
  "shell_probe_output": "Python 3.11.14",
  "shell_probe_exit_code": 0
}


### Backend Parameters in This Lab

The line `BACKEND = Lab4SandboxBackend(...)` is where the Deep Agents backend becomes concrete.

Parameters used here:

- `root_dir=str(SANDBOX_ROOT)`
  The host folder that backs the workspace. In this lab that is `lab_4_deep_agents/sandbox`.
- `virtual_mode=True`
  File tools treat the sandbox as `/`, so `/professors.md` resolves inside the sandbox. It also blocks file-path tricks like `..` and `~`.
  Important: with `LocalShellBackend`, this does **not** sandbox arbitrary shell commands by itself.
- `timeout=300`
  Default maximum time, in seconds, for `execute(...)`.
- `max_output_bytes=100_000`
  Caps captured shell output so one command cannot flood the notebook trace.
- `env=SHELL_ENV`
  Explicit environment variables passed to shell execution. In this lab that includes the Python path plus the LLM and OCR settings needed by the add script.
- `inherit_env=False`
  Do not inherit the whole notebook or host environment automatically. Only the variables we explicitly pass are available.

Sandboxing in Lab 4 is stronger than one parameter:

- `root_dir` points the backend at the sandbox folder
- `virtual_mode=True` makes file-tool paths behave like a virtual `/`
- custom `execute()` blocks every shell command except the sanctioned add flow
- the main prompt and subagent prompt repeat the same boundary in natural language


In [8]:
pretty(
    {
        "backend_parameters": {
            "root_dir": {
                "value": str(SANDBOX_ROOT),
                "meaning": "host folder that backs the workspace",
            },
            "virtual_mode": {
                "value": True,
                "meaning": "file tools treat the sandbox as `/` and block path traversal",
                "important_note": "does not sandbox arbitrary shell execution by itself",
            },
            "timeout": {
                "value": 300,
                "meaning": "default execute timeout in seconds",
            },
            "max_output_bytes": {
                "value": 100_000,
                "meaning": "truncate oversized shell output",
            },
            "env": {
                "value": sorted(SHELL_ENV.keys()),
                "meaning": "explicit environment passed to shell commands",
            },
            "inherit_env": {
                "value": False,
                "meaning": "do not inherit the full host environment automatically",
            },
        },
        "sandboxing_layers": {
            "filesystem_boundary": str(SANDBOX_ROOT),
            "virtual_root": "/",
            "allowed_shell_probe": "python --version",
            "allowed_add_command_pattern": ALLOWED_ADD_COMMAND.pattern,
        },
    }
)


{
  "backend_parameters": {
    "root_dir": {
      "value": "/Users/khajievroma/Projects/agents_tutorial/lab_4_deep_agents/sandbox",
      "meaning": "host folder that backs the workspace"
    },
    "virtual_mode": {
      "value": true,
      "meaning": "file tools treat the sandbox as `/` and block path traversal",
      "important_note": "does not sandbox arbitrary shell execution by itself"
    },
    "timeout": {
      "value": 300,
      "meaning": "default execute timeout in seconds"
    },
    "max_output_bytes": {
      "value": 100000,
      "meaning": "truncate oversized shell output"
    },
    "env": {
      "value": [
        "BIT_PROF_LLM_API_KEY",
        "BIT_PROF_LLM_BASE_URL",
        "BIT_PROF_LLM_MODEL",
        "BIT_PROF_OCR_API_KEY",
        "BIT_PROF_OCR_BASE_URL",
        "BIT_PROF_OCR_MODEL",
        "HOME",
        "PATH",
        "PYTHONPATH"
      ],
      "meaning": "explicit environment passed to shell commands"
    },
    "inherit_env": {
      "value"

## 6. Stream the Deep Agent in Gradio

Learning goal: inspect the Deep Agent in one chat surface, with the live answer and the decision trace rendered inline.

This inline app keeps the full Lab 4 behavior:

- broad questions stay grounded in `/professors.md`
- named-professor questions can open sandbox dossiers under `/professors/`
- add-from-URL requests can delegate to `AddProfessorSubagent`
- tool calls and tool results appear as inline boxes inside the chat
- the live assistant reply stays visible as one normal message bubble while the loop runs
- the trace shows only exposed Deep Agents stream data, not hidden reasoning

`Start a new chat` resets the current `thread_id`, clears the UI, and prepares the current sandbox workspace in place. That rebuilds `professors.md` and refreshes the live add skill without restoring dossier files that were deleted from `/professors/`.


In [9]:
import time
import uuid

import gradio as gr

VISIBLE_REPLY_PLACEHOLDER = "_Working in sandbox..._"

GRADIO_EXAMPLES = [
    "Which professors should I look at for human-computer interaction and virtual environments?",
    "Tell me more about Cheng Cheng.",
    f"Add the professor from this official BIT CSAT detail URL to the workspace: {EXAMPLE_ADD_DETAIL_URL}",
]


def content_to_text(content: Any) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts: list[str] = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                text = item.get("text") or item.get("content")
                parts.append(str(text) if text else json.dumps(item, ensure_ascii=False))
            else:
                parts.append(str(item))
        return "".join(parts)
    return str(content)


def json_safe(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if hasattr(value, "model_dump"):
        try:
            return json_safe(value.model_dump())
        except Exception:
            pass
    return str(value)


def shorten_text(text: str, limit: int = 1200) -> str:
    normalized = text.strip()
    if not normalized:
        return "(empty)"
    if len(normalized) <= limit:
        return normalized
    return normalized[:limit].rstrip() + "\n... (truncated)"


def format_block(value: Any) -> str:
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return "_Waiting for more streamed content..._"
        try:
            parsed = json.loads(stripped)
        except Exception:
            rendered = shorten_text(stripped)
            return f"```text\n{rendered}\n```"
        return f"```json\n{json.dumps(parsed, ensure_ascii=False, indent=2)}\n```"
    return f"```json\n{json.dumps(json_safe(value), ensure_ascii=False, indent=2)}\n```"


def build_thread_id() -> str:
    return f"lab4-gradio-{uuid.uuid4().hex[:8]}"


def next_trace_id(session: dict[str, Any], prefix: str) -> str:
    session["trace_counter"] += 1
    return f"{prefix}-{session['trace_counter']}"


def new_gradio_session() -> dict[str, Any]:
    # Prepare the current sandbox in place for each fresh Gradio session.
    prepare_runtime_workspace()
    thread_id = build_thread_id()
    return {
        "thread_id": thread_id,
        "chat_messages": [],
        "trace_counter": 0,
        "tool_calls": {},
        "tool_call_slots": {},
        "current_run": None,
    }


def find_message_index(chat_messages: list[dict[str, Any]], trace_id: str) -> int:
    for index, message in enumerate(chat_messages):
        metadata = dict(message.get("metadata") or {})
        if metadata.get("id") == trace_id:
            return index
    return -1


def visible_reply_index(chat_messages: list[dict[str, Any]]) -> int:
    for index in range(len(chat_messages) - 1, -1, -1):
        message = chat_messages[index]
        metadata = dict(message.get("metadata") or {})
        if message.get("role") == "assistant" and not metadata.get("title"):
            return index
    return -1


def upsert_trace_message(
    chat_messages: list[dict[str, Any]],
    *,
    trace_id: str,
    title: str,
    content: str,
    parent_id: str | None = None,
    status: str | None = None,
    log: str | None = None,
) -> list[dict[str, Any]]:
    updated_messages = list(chat_messages)
    metadata: dict[str, Any] = {"title": title, "id": trace_id}
    if parent_id is not None:
        metadata["parent_id"] = parent_id
    if status is not None:
        metadata["status"] = status
    if log:
        metadata["log"] = log

    message = {
        "role": "assistant",
        "content": content,
        "metadata": metadata,
    }
    message_index = find_message_index(updated_messages, trace_id)
    if message_index == -1:
        reply_index = visible_reply_index(updated_messages)
        if reply_index == -1:
            updated_messages.append(message)
        else:
            updated_messages.insert(reply_index, message)
    else:
        updated_messages[message_index] = message
    return updated_messages


def ensure_visible_reply(chat_messages: list[dict[str, Any]], content: str = VISIBLE_REPLY_PLACEHOLDER) -> list[dict[str, Any]]:
    updated_messages = list(chat_messages)
    reply_index = visible_reply_index(updated_messages)
    if reply_index == -1:
        updated_messages.append({"role": "assistant", "content": content})
        return updated_messages
    if not str(updated_messages[reply_index].get("content", "")).strip():
        updated_messages[reply_index] = {
            **updated_messages[reply_index],
            "content": content,
        }
    return updated_messages


def append_assistant_text(chat_messages: list[dict[str, Any]], text: str) -> list[dict[str, Any]]:
    updated_messages = ensure_visible_reply(chat_messages)
    reply_index = visible_reply_index(updated_messages)
    reply_message = updated_messages[reply_index]
    current_content = str(reply_message.get("content", ""))
    next_content = text if current_content == VISIBLE_REPLY_PLACEHOLDER else current_content + text
    updated_messages[reply_index] = {
        **reply_message,
        "content": next_content,
    }
    return updated_messages


def replace_assistant_text(chat_messages: list[dict[str, Any]], text: str) -> list[dict[str, Any]]:
    updated_messages = ensure_visible_reply(chat_messages)
    reply_index = visible_reply_index(updated_messages)
    updated_messages[reply_index] = {
        **updated_messages[reply_index],
        "content": text,
    }
    return updated_messages


def read_thread_state(thread_id: str) -> dict[str, Any]:
    config = {"configurable": {"thread_id": thread_id}}
    try:
        snapshot = professor_workspace_agent.get_state(config)
    except Exception as exc:
        return {
            "thread_id": thread_id,
            "error": str(exc),
            "stored_messages": [],
            "state_values": {},
            "final_reply": "",
        }

    values = dict(getattr(snapshot, "values", {}) or {})
    stored_messages = list(values.get("messages", []))
    serialized_messages: list[dict[str, Any]] = []
    final_reply = ""
    for message in stored_messages:
        tool_calls = json_safe(getattr(message, "tool_calls", None) or [])
        serialized = {
            "type": type(message).__name__,
            "content": content_to_text(getattr(message, "content", "")),
            "name": getattr(message, "name", None),
            "tool_calls": tool_calls,
            "tool_call_id": getattr(message, "tool_call_id", None),
        }
        serialized["content"] = content_to_text(getattr(message, "content", ""))
        serialized_messages.append(serialized)
        if serialized["type"] == "AIMessage" and not tool_calls and serialized["content"].strip():
            final_reply = serialized["content"].strip()
    return {
        "thread_id": thread_id,
        "stored_messages": serialized_messages,
        "stored_message_count": len(serialized_messages),
        "state_values": json_safe({key: value for key, value in values.items() if key != "messages"}),
        "final_reply": final_reply,
    }


def build_run_trace_content(session: dict[str, Any]) -> str:
    run = dict(session.get("current_run") or {})
    elapsed = time.time() - float(run.get("started_at", time.time()))
    lines = [
        "The assistant reply is the normal message bubble below.",
        "Expand the `TOOL CALL` boxes to inspect the loop.",
        "",
        f"Sandbox root: `/`",
        f"Current agent: `{run.get('last_agent_name') or 'ProfessorWorkspaceAgent'}`",
        f"Run state: `{run.get('run_state', 'idle')}`",
    ]
    if run.get("latest_action"):
        lines.append(f"Latest loop step: `{run['latest_action']}`")
    if run.get("run_state") in {"complete", "failed"}:
        lines.append(f"Elapsed: `{elapsed:.1f}s`")
        lines.append(f"Tool calls: `{len(run.get('tool_call_ids', []))}`")
        if run.get("stored_message_count") is not None:
            lines.append(f"Stored LangGraph messages: `{run['stored_message_count']}`")
    return "\n".join(lines)


def sync_run_trace(chat_messages: list[dict[str, Any]], session: dict[str, Any], *, thread_state: dict[str, Any] | None = None) -> list[dict[str, Any]]:
    run = dict(session.get("current_run") or {})
    if not run:
        return chat_messages
    root_id = run["root_id"]
    if thread_state is not None:
        run["stored_message_count"] = thread_state.get("stored_message_count")
        session["current_run"] = run
    chat_messages = upsert_trace_message(
        chat_messages,
        trace_id=root_id,
        title="AGENT LOOP",
        content=build_run_trace_content(session),
        status="done" if run.get("run_state") in {"complete", "failed"} else "pending",
        log=run.get("latest_action") or run.get("last_agent_name") or "ProfessorWorkspaceAgent",
    )
    return chat_messages


def absorb_tool_call_chunks(
    chat_messages: list[dict[str, Any]],
    session: dict[str, Any],
    *,
    agent_name: str,
    tool_call_chunks: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    run = dict(session.get("current_run") or {})
    root_id = run["root_id"]

    for chunk in tool_call_chunks:
        slot_key = f"{agent_name}:{chunk.get('index', 0)}"
        call_id = chunk.get("id") or session["tool_call_slots"].get(slot_key)
        if not call_id:
            continue
        if chunk.get("id"):
            session["tool_call_slots"][slot_key] = chunk["id"]

        tool_call = dict(session["tool_calls"].get(call_id) or {})
        if not tool_call:
            trace_id = next_trace_id(session, "tool-call")
            run["tool_call_ids"].append(call_id)
            tool_call = {
                "trace_id": trace_id,
                "agent_name": agent_name,
                "name": None,
                "args": "",
                "id": call_id,
                "result_trace_id": None,
            }

        if chunk.get("name"):
            tool_call["name"] = chunk["name"]
            run["latest_action"] = f"{agent_name} is preparing {chunk['name']}"
        if chunk.get("args"):
            tool_call["args"] = str(tool_call.get("args", "")) + str(chunk["args"])
        session["tool_calls"][call_id] = tool_call

        chat_messages = upsert_trace_message(
            chat_messages,
            trace_id=tool_call["trace_id"],
            title=f"TOOL CALL · {tool_call.get('name') or 'pending'}",
            content="\n\n".join(
                [
                    f"Agent: `{tool_call['agent_name']}`",
                    f"Call id: `{tool_call['id']}`",
                    "Arguments:",
                    format_block(str(tool_call.get("args", ""))),
                ]
            ),
            parent_id=root_id,
            status="pending",
        )
    session["current_run"] = run
    return chat_messages


def record_tool_result(
    chat_messages: list[dict[str, Any]],
    session: dict[str, Any],
    *,
    agent_name: str,
    tool_call_id: str | None,
    tool_name: str | None,
    result_text: str,
) -> list[dict[str, Any]]:
    run = dict(session.get("current_run") or {})
    root_id = run["root_id"]
    resolved_call_id = tool_call_id or next_trace_id(session, "tool-fallback")
    tool_call = dict(session["tool_calls"].get(resolved_call_id) or {})

    if not tool_call:
        trace_id = next_trace_id(session, "tool-call")
        run["tool_call_ids"].append(resolved_call_id)
        tool_call = {
            "trace_id": trace_id,
            "agent_name": agent_name,
            "name": tool_name or "tool",
            "args": "",
            "id": resolved_call_id,
            "result_trace_id": None,
        }

    if tool_name and not tool_call.get("name"):
        tool_call["name"] = tool_name
    session["tool_calls"][resolved_call_id] = tool_call
    run["latest_action"] = f"{agent_name} received result from {tool_call.get('name') or tool_name or 'tool'}"

    chat_messages = upsert_trace_message(
        chat_messages,
        trace_id=tool_call["trace_id"],
        title=f"TOOL CALL · {tool_call.get('name') or 'tool'}",
        content="\n\n".join(
            [
                f"Agent: `{tool_call['agent_name']}`",
                f"Call id: `{tool_call['id']}`",
                "Arguments:",
                format_block(str(tool_call.get("args", ""))),
            ]
        ),
        parent_id=root_id,
        status="done",
    )

    if not tool_call.get("result_trace_id"):
        tool_call["result_trace_id"] = next_trace_id(session, "tool-result")
        run["tool_result_count"] += 1

    chat_messages = upsert_trace_message(
        chat_messages,
        trace_id=tool_call["result_trace_id"],
        title=f"TOOL RESULT · {tool_call.get('name') or tool_name or 'tool'}",
        content=format_block(result_text),
        parent_id=tool_call["trace_id"],
        status="done",
    )
    session["tool_calls"][resolved_call_id] = tool_call
    session["current_run"] = run
    return chat_messages


def record_custom_event(chat_messages: list[dict[str, Any]], session: dict[str, Any], data: Any) -> list[dict[str, Any]]:
    run = dict(session.get("current_run") or {})
    run["custom_event_count"] += 1
    run["latest_action"] = f"custom event {run['custom_event_count']}"
    session["current_run"] = run
    return upsert_trace_message(
        chat_messages,
        trace_id=next_trace_id(session, "custom-event"),
        title=f"EVENT · {run['custom_event_count']}",
        content=format_block(data),
        parent_id=run["root_id"],
        status="done",
    )


def publish_session(session: dict[str, Any], chat_messages: list[dict[str, Any]]):
    session["chat_messages"] = chat_messages
    return "", chat_messages, session


def ask_lab4_agent(prompt: str, session_state: dict[str, Any] | None):
    session = dict(session_state or new_gradio_session())
    prompt = prompt.strip()
    if not prompt:
        yield publish_session(session, session["chat_messages"])
        return

    thread_id = session["thread_id"]
    config = {"configurable": {"thread_id": thread_id}}
    session["tool_calls"] = {}
    session["tool_call_slots"] = {}
    root_id = next_trace_id(session, "run")
    session["current_run"] = {
        "root_id": root_id,
        "started_at": time.time(),
        "run_state": "streaming",
        "chunk_count": 0,
        "message_chunk_count": 0,
        "text_chunk_count": 0,
        "assistant_character_count": 0,
        "tool_call_ids": [],
        "tool_result_count": 0,
        "custom_event_count": 0,
        "last_chunk_type": None,
        "last_agent_name": "ProfessorWorkspaceAgent",
        "latest_action": "waiting for the first tool or reply token",
        "stored_message_count": None,
    }

    chat_messages = [*session["chat_messages"], {"role": "user", "content": prompt}]
    chat_messages = ensure_visible_reply(chat_messages)
    chat_messages = sync_run_trace(chat_messages, session)
    yield publish_session(session, chat_messages)

    try:
        for chunk in professor_workspace_agent.stream(
            {"messages": [{"role": "user", "content": prompt}]},
            config=config,
            stream_mode=["updates", "messages", "custom"],
            subgraphs=True,
            version="v2",
        ):
            run = dict(session.get("current_run") or {})
            run["chunk_count"] += 1
            run["last_chunk_type"] = chunk.get("type")
            session["current_run"] = run

            if chunk["type"] == "messages":
                run["message_chunk_count"] += 1
                token, metadata = chunk["data"]
                agent_name = (
                    metadata.get("lc_agent_name")
                    if isinstance(metadata, dict) and metadata.get("lc_agent_name")
                    else "ProfessorWorkspaceAgent"
                )
                run["last_agent_name"] = agent_name
                token_text = content_to_text(getattr(token, "content", ""))
                node_name = metadata.get("langgraph_node") if isinstance(metadata, dict) else None

                if node_name == "model" and token_text:
                    run["text_chunk_count"] += 1
                    run["assistant_character_count"] += len(token_text)
                    run["latest_action"] = f"{agent_name} is writing the answer"
                    chat_messages = append_assistant_text(chat_messages, token_text)

                tool_call_chunks = list(getattr(token, "tool_call_chunks", None) or [])
                if tool_call_chunks:
                    chat_messages = absorb_tool_call_chunks(
                        chat_messages,
                        session,
                        agent_name=agent_name,
                        tool_call_chunks=tool_call_chunks,
                    )

                if node_name == "tools":
                    chat_messages = record_tool_result(
                        chat_messages,
                        session,
                        agent_name=agent_name,
                        tool_call_id=getattr(token, "tool_call_id", None),
                        tool_name=getattr(token, "name", None),
                        result_text=token_text,
                    )
            elif chunk["type"] == "custom":
                chat_messages = record_custom_event(chat_messages, session, chunk.get("data"))

            chat_messages = sync_run_trace(chat_messages, session)
            yield publish_session(session, chat_messages)

        thread_state = read_thread_state(thread_id)
        final_message = thread_state.get("final_reply", "")
        if final_message:
            chat_messages = replace_assistant_text(chat_messages, final_message)

        run = dict(session.get("current_run") or {})
        run["run_state"] = "complete"
        session["current_run"] = run
        chat_messages = sync_run_trace(chat_messages, session, thread_state=thread_state)
        yield publish_session(session, chat_messages)
    except Exception as exc:
        error_message = f"Lab 4 deep agent stream failed: {exc}"
        run = dict(session.get("current_run") or {})
        run["run_state"] = "failed"
        run["last_chunk_type"] = "error"
        session["current_run"] = run
        chat_messages = replace_assistant_text(chat_messages, error_message)
        chat_messages = sync_run_trace(
            chat_messages,
            session,
            thread_state={"thread_id": thread_id, "error": error_message, "stored_messages": [], "state_values": {}, "final_reply": ""},
        )
        yield publish_session(session, chat_messages)


def reset_lab4_chat():
    session = new_gradio_session()
    return publish_session(session, session["chat_messages"])


initial_session = new_gradio_session()

with gr.Blocks(fill_width=True) as demo:
    gr.Markdown(
        """### Lab 4 Gradio App
Ask the notebook-local Deep Agent anything about the sandbox professor workspace. Each fresh chat prepares the current sandbox in place, rebuilding `professors.md` and restoring the live add skill without repopulating deleted dossier files. The plain assistant bubble stays visible as the live reply, and the loop itself appears as expandable `TOOL CALL` and `TOOL RESULT` boxes in the same chat."""
    )

    session_state = gr.State(initial_session)
    chatbot = gr.Chatbot(
        label="Deep Agent conversation",
        height=720,
        value=initial_session["chat_messages"],
        group_consecutive_messages=False,
    )
    prompt = gr.Textbox(
        label="Ask the Lab 4 Deep Agent",
        placeholder="Try a broad question, a named-professor question, or the add-from-URL workflow.",
    )
    with gr.Row():
        ask_button = gr.Button("Ask", variant="primary")
        reset_button = gr.Button("Start a new chat")
    gr.Examples(examples=GRADIO_EXAMPLES, inputs=prompt)

    ask_button.click(
        ask_lab4_agent,
        inputs=[prompt, session_state],
        outputs=[prompt, chatbot, session_state],
    )
    prompt.submit(
        ask_lab4_agent,
        inputs=[prompt, session_state],
        outputs=[prompt, chatbot, session_state],
    )
    reset_button.click(
        reset_lab4_chat,
        outputs=[prompt, chatbot, session_state],
    )

demo.launch(inline=False, inbrowser=True, height=980, width="100%", prevent_thread_lock=True)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
